In [1]:
%pip install torch transformers pandas scikit-learn tqdm speechrecognition pydub

In [3]:
# ----------------------------
# Install dependencies (run once)
# ----------------------------

# ----------------------------
# Imports
# ----------------------------
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ----------------------------
# Load pre-trained BERT tokenizer and model
# ----------------------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# ----------------------------
# Dataset class
# ----------------------------
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=500):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# ----------------------------
# Load training + validation data
# ----------------------------
df = pd.read_csv('domestic_violence_data.csv')  # Training + validation
texts = df['Text'].tolist()
labels = df['Label'].tolist()

# Split: 70% train, 30% validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

# ----------------------------
# Load independent test data
# ----------------------------
test_df = pd.read_csv('text_test.csv')  # Testing
test_texts = test_df['text'].tolist()
test_labels = test_df['label'].tolist()

# ----------------------------
# Create datasets and dataloaders
# ----------------------------
train_dataset = TextDataset(train_texts, train_labels, tokenizer)
val_dataset = TextDataset(val_texts, val_labels, tokenizer)
test_dataset = TextDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ----------------------------
# Training setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

# ----------------------------
# Training loop with validation
# ----------------------------
for epoch in range(epochs):
    print(f'\nEpoch {epoch+1}/{epochs}')
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        train_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f"Training Loss: {train_loss / len(train_loader):.4f}")

    # Validation
    model.eval()
    val_preds, val_labels_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels_list, val_preds)
    print(f'Validation Accuracy: {val_acc*100:.2f}%')

# ----------------------------
# Test set evaluation
# ----------------------------
model.eval()
test_preds, test_labels_list = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels_list.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels_list, test_preds)
print(f'\nTest Accuracy: {test_acc*100:.2f}%')

# ----------------------------
# Save model and tokenizer for Flask app
# ----------------------------
model.save_pretrained('saved_model')
tokenizer.save_pretrained('saved_model')
print("Model and tokenizer saved in 'saved_model/'")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Vinay B R\anaconda3\anaconda\Lib\site-packages\transformers\optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



Epoch 1/3


Training: 100%|██████████| 5/5 [03:02<00:00, 36.57s/it]


Training Loss: 0.6676
Validation Accuracy: 96.88%

Epoch 2/3


Training: 100%|██████████| 5/5 [03:21<00:00, 40.30s/it]


Training Loss: 0.4840
Validation Accuracy: 96.88%

Epoch 3/3


Training: 100%|██████████| 5/5 [03:15<00:00, 39.10s/it]


Training Loss: 0.3117
Validation Accuracy: 96.88%

Test Accuracy: 96.67%
Model and tokenizer saved in 'saved_model/'
